# Blog figures — every `[B#]` asset for BLOG_DRAFT.md

Companion to the `sanity_0*` notebooks: they *verify* the numbers, this one
*exports the figures*. Runs where they run — on the workspace, against
`$FM_BASE` (default `/mnt/user_storage/transaction-fm-v2`). Every loader
skips loudly if its artifact is missing, so partial runs are fine.

| cell | asset (BLOG_ASSETS.md) | source artifact |
|---|---|---|
| hero | **B1** two-panel dot+CI | `downstream/<sc>/bootstrap_ci.json` + `<sc>_fulltest/` |
| fulltest | **B7** table-2 dot+CI | `downstream/<sc>_fulltest/bootstrap_ci.json` + paired json |
| reco | **B-RECO-FIG** slices bars | `downstream/full_nextmerchant/probe_metrics_v3.json` |
| burst | **B8** ledgered stat + **B9** histogram | `raw/full/transactions.parquet` + `splits.json` (computed here) |
| canary | **B16** month-canary curves | `$BASE/tensorboard/fm_<sc>_*` event files |
| tables | **B10** velocity + final eyeball diff | `full_velocity/velocity_metrics.json`, both CI sets |

Outputs land in `$FM_FIG_OUT` (default `$FM_BASE/figures/`) as PNG (200dpi) + SVG,
plus `burst_stats.json` — the B8 ledger the draft can cite.

*Not producible here:* B2 (architecture graphic — design task), B15 (console job
costs), B11 (optional, needs research-branch artifacts), Zach sign-offs.

In [ ]:
import os, json, glob
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

BASE = os.environ.get("FM_BASE", "/mnt/user_storage/transaction-fm-v2")
FIG_OUT = os.environ.get("FM_FIG_OUT", f"{BASE}/figures")
os.makedirs(FIG_OUT, exist_ok=True)

# Palette (validated): one accent for the story (our embedding), muted ink for
# every other model, an ordinal blue ramp for the three context lengths.
# Highlight-one-gray-the-rest; selective direct labels on the accent series.
ACCENT, MUTED = "#2a78d6", "#898781"
INK, INK2 = "#0b0b0b", "#52514e"
GRID, AXIS, SURFACE = "#e1e0d9", "#c3c2b7", "#fcfcfb"
CTX_RAMP = {512: "#86b6ef", 1024: "#2a78d6", 2048: "#104281"}

SCALES = [("full", 512), ("xl", 1024), ("xxl", 2048)]
NV_BASELINE, NV_FUSION = 0.1238, 0.1755   # their published points

mpl.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": AXIS, "axes.linewidth": 0.8,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6, "axes.axisbelow": True,
    "xtick.color": INK2, "ytick.color": INK2, "text.color": INK,
    "axes.labelcolor": INK2, "font.size": 10.5, "font.family": "sans-serif",
    "axes.spines.top": False, "axes.spines.right": False,
})

def save(fig, name):
    for ext, kw in (("png", {"dpi": 200}), ("svg", {})):
        fig.savefig(f"{FIG_OUT}/{name}.{ext}", bbox_inches="tight", **kw)
    print(f"[saved] {FIG_OUT}/{name}.png + .svg")

def load_results(path):
    if not os.path.exists(path):
        print(f"[SKIP] missing {path}")
        return None
    return json.load(open(path))["results"]

ci100k = {sc: load_results(f"{BASE}/downstream/{sc}/bootstrap_ci.json") for sc, _ in SCALES}
cifull = {sc: load_results(f"{BASE}/downstream/{sc}_fulltest/bootstrap_ci.json") for sc, _ in SCALES}

## B1 — hero: two panels, dot + 95% CI

Left = their 100k protocol (the only NVIDIA-comparable set), with their two
published numbers as reference lines. Right = full test period. Shared y-axis;
the caption carries the never-compare-across-panels warning. Our
`embed_xgb` rows wear the accent; every other readout is context.

In [ ]:
def draw_panel(ax, rows, title):
    """rows: list of (label, dict-from-bootstrap_ci, is_ours)."""
    for x, (label, r, ours) in enumerate(rows):
        if r is None:
            continue
        lo, hi = r["ap_ci95"]
        c = ACCENT if ours else MUTED
        ax.errorbar(x, r["ap"], yerr=[[r["ap"] - lo], [hi - r["ap"]]],
                    fmt="o", color=c, ecolor=c, ms=6.5, capsize=3, lw=1.4, zorder=3)
        if ours:  # selective direct labels: only the story series
            ax.annotate(f'{r["ap"]:.3f}', (x, hi), textcoords="offset points",
                        xytext=(0, 5), ha="center", fontsize=9, color=INK)
    ax.set_xticks(range(len(rows)))
    ax.set_xticklabels([l for l, _, _ in rows], rotation=30, ha="right", fontsize=9)
    ax.set_title(title, fontsize=11, color=INK, pad=10)

if all(ci100k[sc] for sc, _ in SCALES) and all(cifull[sc] for sc, _ in SCALES):
    fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.6), sharey=True)

    left = [
        ("raw 13 feats (baseline)", ci100k["full"].get("baseline"),        False),
        ("their PCA64 · 512",       ci100k["full"].get("embed_pca64_xgb"), False),
        ("their PCA64 · 1024",      ci100k["xl"].get("embed_pca64_xgb"),   False),
        ("linear head · 512",       ci100k["full"].get("embed_logistic"),  False),
        ("embed→XGB · 512",         ci100k["full"].get("embed_xgb"),       True),
        ("embed→XGB · 1024",        ci100k["xl"].get("embed_xgb"),         True),
        ("embed→XGB · 2048",        ci100k["xxl"].get("embed_xgb"),        True),
    ]
    draw_panel(axes[0], left, "Their protocol — 100k test (112 frauds)")
    axes[0].axhline(NV_FUSION, color=INK2, lw=1.1, ls=(0, (5, 3)), zorder=2)
    axes[0].axhline(NV_BASELINE, color=MUTED, lw=0.9, ls=(0, (5, 3)), zorder=2)
    axes[0].annotate(f"NVIDIA published fusion  {NV_FUSION}", (0.02, NV_FUSION),
                     xycoords=("axes fraction", "data"), va="bottom", fontsize=9, color=INK2)
    axes[0].annotate(f"NVIDIA published baseline  {NV_BASELINE}", (0.02, NV_BASELINE),
                     xycoords=("axes fraction", "data"), va="bottom", fontsize=9, color=MUTED)
    axes[0].set_ylabel("Average precision (test)")

    right = [
        ("raw 13 feats (baseline)", cifull["full"].get("baseline"),  False),
        ("embed→XGB · 512",         cifull["full"].get("embed_xgb"), True),
        ("embed→XGB · 1024",        cifull["xl"].get("embed_xgb"),   True),
        ("embed→XGB · 2048 (40ep)", cifull["xxl"].get("embed_xgb"),  True),
    ]
    draw_panel(axes[1], right, "Full test period — 2.44M rows (2,724 frauds)")

    fig.suptitle("The embedding alone clears their published fusion headline — "
                 "and the lift is CI-separated on the full test period",
                 fontsize=12.5, color=INK, y=1.04)
    fig.text(0.5, -0.06, "Dots = point AP, whiskers = bootstrap 95% CI. The two panels are "
             "different eval sets — never compare numbers across them.",
             ha="center", fontsize=9.5, color=INK2)
    save(fig, "b1_hero")
    plt.show()
else:
    print("[SKIP] B1 — need bootstrap_ci.json for all scales (100k + fulltest)")

## B7 — table 2 as a dot-and-CI plot

Every fulltest row; the raw baseline's CI drawn as a reference band so
"CI-separated" is visible rather than asserted. Paired-ordering stats from
`paired_bootstrap_embed_xgb.json` go in the caption.

In [ ]:
pb_path = f"{BASE}/downstream/paired_bootstrap_embed_xgb.json"
pb = json.load(open(pb_path)) if os.path.exists(pb_path) else None

if all(cifull[sc] for sc, _ in SCALES):
    rows = [
        ("their PCA64 · 512",       cifull["full"].get("embed_pca64_xgb"), False),
        ("their PCA64 · 1024",      cifull["xl"].get("embed_pca64_xgb"),   False),
        ("linear head · 512",       cifull["full"].get("embed_logistic"),  False),
        ("linear head · 1024",      cifull["xl"].get("embed_logistic"),    False),
        ("embed→XGB · 512",         cifull["full"].get("embed_xgb"),       True),
        ("embed→XGB · 1024",        cifull["xl"].get("embed_xgb"),         True),
        ("embed→XGB · 2048 (40ep)", cifull["xxl"].get("embed_xgb"),        True),
    ]
    base_r = cifull["full"]["baseline"]

    fig, ax = plt.subplots(figsize=(9.5, 4.8))
    ax.axhspan(*base_r["ap_ci95"], color=GRID, alpha=0.55, zorder=1)
    ax.axhline(base_r["ap"], color=INK2, lw=1.0, zorder=2)
    ax.annotate(f'raw-13 baseline  {base_r["ap"]:.3f} (95% CI band)',
                (0.985, base_r["ap"]), xycoords=("axes fraction", "data"),
                ha="right", va="bottom", fontsize=9, color=INK2)
    draw_panel(ax, rows, "Full test period (2.44M rows): all models on identical rows")
    ax.set_ylabel("Average precision (test)")

    cap = "Dots = point AP, whiskers = bootstrap 95% CI."
    if pb:
        ctx_name = {"full_fulltest": "512", "xl_fulltest": "1024", "xxl_fulltest": "2048"}
        pieces = []
        for k, v in pb["pairs"].items():
            a, b = [ctx_name.get(s.strip(), s.strip()) for s in k.split(" - ")]
            pieces.append(f'{a}−{b}: {v["mean_diff"]:+.3f} '
                          f'[{v["ci95"][0]:+.3f}, {v["ci95"][1]:+.3f}] P={v["p_a_gt_b"]:.3f}')
        cap += "  Paired bootstrap (same resampled rows): " + " · ".join(pieces)
    fig.text(0.5, -0.08, cap, ha="center", fontsize=8.8, color=INK2, wrap=True)
    save(fig, "b7_fulltest")
    plt.show()
else:
    print("[SKIP] B7 — need fulltest bootstrap_ci.json for all scales")

## B-RECO-FIG — where the FM earns its budget

Grouped bars, HR@10: overall (honest memorization floor vs the hybrid), the
next-merchant∉top-10 slice (memorization structurally 0), and never-seen
merchants (any history method structurally 0). Accent = FM-powered method
(hybrid overall, full-vocab MLP on the slices — labeled per group).

In [ ]:
v3p = f"{BASE}/downstream/full_nextmerchant/probe_metrics_v3.json"
if os.path.exists(v3p):
    v3 = json.load(open(v3p))
    sl = v3["slices"]
    groups = [
        ("Overall\n(400k events)",
         v3["readouts"]["naive_count"]["hr@10"], v3["readouts"]["alpha_blend"]["hr@10"],
         "hybrid: 0.1·MLP + 0.9·freq"),
        (f"Next merchant NOT in card's top-10\n({sl['share_not_in_top10']:.0%} of events)",
         sl["not_in_top10"]["naive_count"], sl["not_in_top10"]["mlp_solo_token_space"],
         "full-vocab MLP"),
        (f"Never-seen merchant\n({sl['share_never_seen_or_beyond_cap']:.0%} of events)",
         sl["never_seen_or_beyond_cap"]["naive_count"],
         sl["never_seen_or_beyond_cap"]["mlp_solo_token_space"], "full-vocab MLP"),
    ]
    x = np.arange(len(groups)); w = 0.36
    fig, ax = plt.subplots(figsize=(9, 4.4))
    bars_naive = ax.bar(x - w / 2, [g[1] for g in groups], w, color=MUTED,
                        edgecolor=SURFACE, linewidth=2,
                        label="memorization baseline (top-10 history)")
    bars_fm = ax.bar(x + w / 2, [g[2] for g in groups], w, color=ACCENT,
                     edgecolor=SURFACE, linewidth=2, label="FM-powered")
    for bars in (bars_naive, bars_fm):
        for rect in bars:
            v = rect.get_height()
            ax.annotate(f"{v:.3f}" if v else "0",
                        (rect.get_x() + rect.get_width() / 2, v),
                        textcoords="offset points", xytext=(0, 4), ha="center",
                        fontsize=9, color=INK)
    for xi, g in zip(x, groups):  # which FM method carries each group
        ax.annotate(g[3], (xi + w / 2, 0), textcoords="offset points", xytext=(0, -34),
                    ha="center", fontsize=8.2, color=INK2)
    ax.set_xticks(x); ax.set_xticklabels([g[0] for g in groups], fontsize=9.5)
    ax.set_ylabel("HR@10"); ax.legend(frameon=False, fontsize=9, loc="upper right")
    ax.set_title("The recommender beats memorization overall — and covers the slices "
                 "memorization is structurally blind to", fontsize=11.5, color=INK, pad=12)
    save(fig, "b_reco_slices")
    plt.show()
else:
    print(f"[SKIP] B-RECO-FIG — missing {v3p}")

## B8 + B9 — the burst stat, computed and ledgered

The draft's "90% of test frauds have a prior same-card fraud within the
preceding 512 transactions vs 7.3% of normals" currently lives in campaign
memory, not an artifact. This computes it from the raw parquet (distance in
same-card transactions to the most recent *strictly prior* fraud), writes
`burst_stats.json` as the citable ledger, and draws the histogram with the
context ceilings (~315 = their token budget, 512/1024/2048 = ours) marked.

In [ ]:
import pyarrow.dataset as pads

raw_p = f"{BASE}/raw/full/transactions.parquet"
splits_p = f"{BASE}/raw/full/splits.json"
if os.path.exists(splits_p):
    splits = json.load(open(splits_p))
    df = (pads.dataset(raw_p, format="parquet")
              .to_table(columns=["card_id", "timestamp", "is_fraud"]).to_pandas())
    df = df.sort_values(["card_id", "timestamp"], kind="stable").reset_index(drop=True)
    print(f"{len(df):,} transactions, {df.is_fraud.sum():,} frauds "
          f"({df.is_fraud.mean():.4%})")

    grp = df.groupby("card_id", sort=False)
    pos = grp.cumcount()
    fraud_pos = pos.where(df.is_fraud.astype(bool))
    # last fraud at-or-before each row, then shift within card -> strictly prior
    prior = fraud_pos.groupby(df.card_id, sort=False).ffill()
    prior = prior.groupby(df.card_id, sort=False).shift(1)
    dist = pos - prior  # NaN = no prior same-card fraud

    val_end = splits["val_end"]
    ts = df["timestamp"]
    if not np.issubdtype(ts.dtype, np.number):
        ts = ts.astype("int64")
        val_end = pd.Timestamp(val_end).value if isinstance(val_end, str) else val_end
    test = ts > val_end
    print(f"test period: {test.sum():,} rows, {df.is_fraud[test].sum():,} frauds")

    is_fraud = df.is_fraud.astype(bool)
    stats = {"definition": "distance in same-card transactions to most recent "
                           "strictly-prior fraud; test-period rows (ts > val_end)",
             "n_test": int(test.sum()), "n_test_fraud": int(is_fraud[test].sum())}
    for grp_name, mask in (("fraud", test & is_fraud), ("normal", test & ~is_fraud)):
        d_grp = dist[mask]
        s = {"share_no_prior_fraud": float(d_grp.isna().mean())}
        for k in (315, 512, 1024, 2048):
            s[f"share_within_{k}"] = float((d_grp <= k).mean())
        stats[grp_name] = s
    with open(f"{FIG_OUT}/burst_stats.json", "w") as f:
        json.dump(stats, f, indent=2)
    print(json.dumps(stats, indent=2))
    print(f"[saved] {FIG_OUT}/burst_stats.json  <- cite these exact numbers in B8")

    fig, ax = plt.subplots(figsize=(9, 4.2))
    bins = np.logspace(0, np.log10(max(float(dist.max()), 4096.0)), 48)
    for name, mask, color, fill in (("normal txns", test & ~is_fraud, MUTED, False),
                                    ("fraud txns", test & is_fraud, ACCENT, True)):
        d_grp = dist[mask].dropna()
        weights = np.full(len(d_grp), 1.0 / mask.sum())  # share of GROUP incl. no-prior rows
        ax.hist(d_grp, bins=bins, weights=weights,
                histtype="stepfilled" if fill else "step",
                color=color, alpha=0.75 if fill else 1.0, lw=1.6, label=name, zorder=3)
    for k, lab in ((315, "~315 — their token budget"), (512, "512"),
                   (1024, "1024"), (2048, "2048")):
        ax.axvline(k, color=AXIS, lw=0.9, ls=(0, (4, 3)), zorder=2)
        ax.annotate(lab, (k, ax.get_ylim()[1]), textcoords="offset points",
                    xytext=(3, -2), fontsize=8.2, color=INK2, va="top")
    ax.set_xscale("log")
    ax.set_xlabel("same-card transactions since previous fraud (log scale)")
    ax.set_ylabel("share of group")
    ax.legend(frameon=False, fontsize=9)
    ax.set_title("Fraud is bursty: the signal sits within the context window",
                 fontsize=11.5, color=INK, pad=10)
    save(fig, "b9_burst")
    plt.show()
else:
    print(f"[SKIP] B8/B9 — missing {splits_p}")

## B16 — the month canary + macro accuracy

`field_ce/month` per optimizer step (same optimizer work = fair x-axis across
context lengths) and `epoch/acc_macro` per epoch, all three scales on the
validated ordinal blue ramp. The 2048 run's 20→40 continuation appears
naturally (its resumed run carries both event streams).

In [ ]:
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

def scalars(scale, tag):
    """Merge a tag across every event file/run dir for a scale, sorted by step."""
    pts = []
    for run in sorted(glob.glob(f"{BASE}/tensorboard/fm_{scale}_*")):
        acc = EventAccumulator(run); acc.Reload()
        if tag in acc.Tags()["scalars"]:
            pts += [(e.step, e.value) for e in acc.Scalars(tag)]
    return sorted(set(pts)) if pts else None

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
found = False
for scale, ctx in SCALES:
    for ax, tag in ((axes[0], "field_ce/month"), (axes[1], "epoch/acc_macro")):
        pts = scalars(scale, tag)
        if pts:
            found = True
            xs, ys = zip(*pts)
            ax.plot(xs, ys, color=CTX_RAMP[ctx], lw=2, label=f"{ctx} ctx", zorder=3)
            ax.annotate(f"{ctx}", (xs[-1], ys[-1]), textcoords="offset points",
                        xytext=(5, 0), fontsize=9, color=CTX_RAMP[ctx], va="center")
if found:
    axes[0].set_title("month canary — field_ce/month (lower = learned)",
                      fontsize=11, color=INK)
    axes[0].set_xlabel("optimizer step"); axes[0].set_ylabel("cross-entropy")
    axes[1].set_title("per-field macro accuracy by epoch", fontsize=11, color=INK)
    axes[1].set_xlabel("epoch"); axes[1].set_ylabel("accuracy")
    axes[0].legend(frameon=False, fontsize=9)
    fig.suptitle("Longer context = fewer windows per epoch: the 512 run hits its month "
                 "phase transition; the longer runs grind toward theirs",
                 fontsize=12, color=INK, y=1.03)
    save(fig, "b16_canary")
    plt.show()
else:
    print(f"[SKIP] B16 — no event files under {BASE}/tensorboard/fm_*")

## B10 + final eyeball diff

The velocity-control numbers for the B10 table, the shuffled-label control,
and both CI tables printed once more — diff these against the draft's inline
tables *before* exporting anything above, so figures and prose can't drift.

In [ ]:
for p, label in (
        (f"{BASE}/downstream/full_velocity/velocity_metrics.json", "B10 velocity control"),
        (f"{BASE}/downstream/full_probe/probe_metrics_shuffled_seed0.json",
         "shuffled-label control")):
    if os.path.exists(p):
        print(f"== {label}: {p}")
        print(json.dumps(json.load(open(p)), indent=2)[:2000])
    else:
        print(f"[SKIP] {label} — missing {p}")

for name, src in (("TABLE 1 (100k)", ci100k), ("TABLE 2 (fulltest)", cifull)):
    rows = []
    for sc, ctx in SCALES:
        if src[sc]:
            for m, r in src[sc].items():
                rows.append({"model": m, "ctx": ctx, "ap": round(r["ap"], 4),
                             "ci": [round(x, 3) for x in r["ap_ci95"]],
                             "P(>fusion)": round(r.get("p_beats_nvidia_fusion",
                                                       float("nan")), 3)})
    if rows:
        print(f"\n== {name} — diff against BLOG_DRAFT.md")
        print(pd.DataFrame(rows).to_string(index=False))

---
**Produced here:** `b1_hero`, `b7_fulltest`, `b_reco_slices`, `b9_burst`,
`b16_canary` (PNG+SVG each) + `burst_stats.json` (the B8 ledger — update the
draft's 90%/7.3% sentence with these exact values).

**Still manual (see BLOG_ASSETS.md):** B2 architecture graphic, B15 console
job costs, B11 (optional), Zach sign-offs + links, venue/CTA decisions.